# A/B Testing Methods: A Comprehensive Guide

## The Gold Standard for Product Decisions

A/B testing is the most reliable way to measure whether a change to your product actually improves user outcomes. Instead of relying on hunches or past patterns, A/B testing lets you **randomly assign users to different versions** and compare results with statistical rigor.

### Why A/B Testing Matters

- **Causality, not correlation**: A/B testing randomly assigns users, so differences between groups are caused by your change, not hidden factors
- **Controlled experiments**: Only one thing changes between the control (original) and treatment (new version)
- **Statistical confidence**: You know how confident you should be in your results, accounting for randomness
- **Risk reduction**: Test changes safely before rolling out to everyone

### The Challenge

A single experiment can be analyzed many different ways, leading to different conclusions. In this notebook, we'll explore **5 major statistical methods** for A/B testing, understand when each works best, and see what they reveal about your data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Define colors for consistency
colors = {
    'primary': '#2E86AB',
    'secondary': '#F18F01',
    'success': '#2CA58D',
    'danger': '#E15554'
}

# Create output directory
import os
os.makedirs('../data/outputs/nb03/', exist_ok=True)

print('Environment set up successfully!')

## Section 1: Load Data & Sanity Checks

Before analyzing results, we verify that our experiment was set up correctly. The most important check is the **Sample Ratio Mismatch (SRM)**: we expect roughly equal users in treatment and control groups.

In [ ]:
# Load data
df = pd.read_csv('../data/inputs/ab_clean.csv')

print('Dataset shape:', df.shape)
print('\nFirst few rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nBasic statistics:')
print(df.describe())

In [ ]:
# Sanity Check 1: Sample Ratio Mismatch (SRM)
group_counts = df['group'].value_counts()
print('Group distribution:')
print(group_counts)
print('\nProportions:')
print(group_counts / len(df))

# Chi-square test for equal split
chi2, p_value = stats.chisquare([group_counts['control'], group_counts['treatment']])
print(f'\nSRM Test (Chi-Square): chi2 = {chi2:.4f}, p-value = {p_value:.4f}')
if p_value > 0.05:
    print('PASS: Groups are balanced (no SRM detected)')
else:
    print('WARNING: Significant imbalance detected')

# Sanity Check 2: Missing values
print('\nMissing values:')
print(df.isnull().sum())

# Sanity Check 3: Conversion rate definition
print('\nConversion column value counts:')
print(df['converted'].value_counts())
print(f'Overall conversion rate: {df["converted"].mean():.2%}')

In [ ]:
# Basic statistics by group
print('=== CONTROL GROUP ===')
control = df[df['group'] == 'control']
print(f'Sample size: {len(control)}')
print(f'Conversions: {control["converted"].sum()}')
print(f'Conversion rate: {control["converted"].mean():.4f}')
print(f'Revenue per user: ${control["revenue"].mean():.2f}')
print(f'Pages viewed: {control["pages_viewed"].mean():.2f}')
print(f'Time on site: {control["time_on_site_seconds"].mean():.1f} seconds')

print('\n=== TREATMENT GROUP ===')
treatment = df[df['group'] == 'treatment']
print(f'Sample size: {len(treatment)}')
print(f'Conversions: {treatment["converted"].sum()}')
print(f'Conversion rate: {treatment["converted"].mean():.4f}')
print(f'Revenue per user: ${treatment["revenue"].mean():.2f}')
print(f'Pages viewed: {treatment["pages_viewed"].mean():.2f}')
print(f'Time on site: {treatment["time_on_site_seconds"].mean():.1f} seconds')

print('\n=== DIFFERENCE ===')
print(f'Conversion rate lift: {(treatment["converted"].mean() - control["converted"].mean()):.4f}')
print(f'Lift %: {((treatment["converted"].mean() / control["converted"].mean()) - 1):.2%}')
print(f'Revenue difference: ${treatment["revenue"].mean() - control["revenue"].mean():.2f}')

## Method 1: Frequentist Z-Test (Two-Proportion Test)

### How It Works

The z-test compares the **conversion rates** between groups using the normal distribution. It asks: "If the treatment had no effect, how unlikely would we be to see this difference?"

**The Math (Simplified):**
1. Calculate conversion rates for each group: p1, p2
2. Pool them to estimate the common rate under "no effect"
3. Calculate standard error (how much we would expect variation)
4. Compute z-statistic: how many standard errors away is our observed difference?
5. Convert z to p-value: probability of seeing this by chance

### Key Concepts

- **p-value < 0.05**: Strong evidence against "no effect" (conventional threshold)
- **95% Confidence Interval**: Range we are 95% confident contains the true difference
- **Effect Size**: Actual magnitude of the difference (lift)

### Benefits
- Fast to compute, intuitive interpretation
- Works well with large samples
- Standard approach in industry

### Limitations
- Assumes normal distribution (reasonable with large samples)
- Can be misleading with small sample sizes
- Does not account for practical significance (a tiny lift might be statistically significant)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Extract conversion data
control_conversions = control['converted'].sum()
control_n = len(control)
treatment_conversions = treatment['converted'].sum()
treatment_n = len(treatment)

print('Conversion counts:')
print(f'Control: {control_conversions}/{control_n}')
print(f'Treatment: {treatment_conversions}/{treatment_n}')

# Perform z-test
count = np.array([treatment_conversions, control_conversions])
nobs = np.array([treatment_n, control_n])

z_stat, p_value_ztest = proportions_ztest(count, nobs, alternative='two-sided')

print(f'\n=== Z-TEST RESULTS ===')
print(f'Z-statistic: {z_stat:.4f}')
print(f'P-value (two-sided): {p_value_ztest:.6f}')
print(f'Significance level: alpha = 0.05')
print(f'Result: {"SIGNIFICANT" if p_value_ztest < 0.05 else "NOT SIGNIFICANT"}')

# Calculate 95% CI for the difference
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(treatment_conversions, treatment_n, 
                                               control_conversions, control_n)

print(f'\n=== CONFIDENCE INTERVAL ===')
print(f'Treatment conversion rate: {treatment_conversions/treatment_n:.4f}')
print(f'Control conversion rate: {control_conversions/control_n:.4f}')
print(f'Difference: {(treatment_conversions/treatment_n) - (control_conversions/control_n):.4f}')
print(f'95% CI for difference: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'\nInterpretation: We are 95% confident the true difference is between {ci_low:.2%} and {ci_high:.2%}')

## Method 2: Chi-Square Test (Test of Independence)

### How It Works

The chi-square test compares **observed vs. expected frequencies** in a contingency table. It asks: "Are conversion and group assignment independent, or related?"

**The Math (Simplified):**
1. Create a 2x2 table: (converted Y/N) x (treatment/control)
2. Calculate expected frequencies under "no relationship"
3. Compare observed to expected: chi-square = sum((observed - expected)^2/expected)
4. Convert chi-square to p-value using chi-square distribution

### Key Concepts

- **Contingency Table**: Shows all combinations of group and outcome
- **chi-square statistic**: Larger values indicate stronger evidence of relationship
- **Degrees of freedom**: (rows-1) x (cols-1) = 1 for our 2x2 table

### Benefits
- No assumptions about normal distribution
- Directly tests independence of categorical variables
- Robust and well-established

### Limitations
- Less intuitive than z-test
- Sensitive to sample size (large N can detect tiny effects)
- Does not give confidence intervals on effect size

In [ ]:
# Create contingency table
contingency_table = pd.crosstab(df['group'], df['converted'])
print('Contingency Table:')
print(contingency_table)
print('\nAs percentages (by group):')
print(pd.crosstab(df['group'], df['converted'], normalize='index'))

# Perform chi-square test
chi2, p_value_chi2, dof, expected = chi2_contingency(contingency_table)

print(f'\n=== CHI-SQUARE TEST RESULTS ===')
print(f'chi-square statistic: {chi2:.4f}')
print(f'P-value: {p_value_chi2:.6f}')
print(f'Degrees of freedom: {dof}')
print(f'Result: {"SIGNIFICANT" if p_value_chi2 < 0.05 else "NOT SIGNIFICANT"}')

print(f'\nExpected frequencies under independence:')
print(pd.DataFrame(expected, 
                   index=['control', 'treatment'], 
                   columns=[0, 1]))

print(f'\n=== INTERPRETATION ===')
if p_value_chi2 < 0.05:
    print(f'We have strong evidence that conversion rate depends on group assignment.')
else:
    print(f'No significant relationship detected between group and conversion.')

## Method 3: Bayesian A/B Testing (Beta-Binomial Model)

### How It Works

Bayesian testing inverts the question: instead of asking "what is the probability of this data under no effect?", we ask **"what is the probability the treatment is better?"**

**The Approach:**
1. Start with a **prior belief**: Beta(1,1) = "complete uncertainty" about true conversion rate
2. Update with **observed data**: Use conjugate Beta distribution (math works out cleanly)
3. Simulate **posterior distributions**: What do we believe about each group is true rate?
4. Compare via Monte Carlo: "In what % of simulations is treatment > control?"

### Key Concepts

- **Prior**: Your starting belief before seeing data
- **Posterior**: Your updated belief after data
- **Credible Interval**: Bayesian version of CI (has more intuitive interpretation)
- **P(treatment > control)**: Direct answer to business question

### Benefits
- Intuitive: "probability treatment is better" directly answers the business question
- Can incorporate prior knowledge
- Does not require pre-planned sample size
- Accounts for uncertainty in both groups

### Limitations
- More computationally complex
- Requires choosing a prior (subjective, though Beta(1,1) is neutral)
- Requires more statistical sophistication to understand

In [ ]:
# Bayesian A/B Testing
np.random.seed(42)

# Prior: Beta(1, 1) = uniform distribution
prior_alpha, prior_beta = 1, 1

# Update with data
control_alpha = prior_alpha + control_conversions
control_beta = prior_beta + (control_n - control_conversions)

treatment_alpha = prior_alpha + treatment_conversions
treatment_beta = prior_beta + (treatment_n - treatment_conversions)

print('Posterior Distributions:')
print(f'Control ~ Beta({control_alpha}, {control_beta})')
print(f'Treatment ~ Beta({treatment_alpha}, {treatment_beta})')

# Monte Carlo simulation
n_samples = 100000
control_samples = np.random.beta(control_alpha, control_beta, n_samples)
treatment_samples = np.random.beta(treatment_alpha, treatment_beta, n_samples)

# Probability treatment > control
prob_treatment_better = (treatment_samples > control_samples).mean()

print(f'\n=== BAYESIAN RESULTS ===')
print(f'P(Treatment > Control) = {prob_treatment_better:.4f} ({prob_treatment_better*100:.2f}%)')

# Credible intervals (94% to match 95% but more intuitive in Bayesian context)
print(f'\nControl - 94% Credible Interval: [{np.percentile(control_samples, 3):.4f}, {np.percentile(control_samples, 97):.4f}]')
print(f'Treatment - 94% Credible Interval: [{np.percentile(treatment_samples, 3):.4f}, {np.percentile(treatment_samples, 97):.4f}]')

# Compute expected values
print(f'\nExpected (posterior mean) conversion rates:')
print(f'Control: {control_samples.mean():.4f}')
print(f'Treatment: {treatment_samples.mean():.4f}')
print(f'Expected lift: {treatment_samples.mean() - control_samples.mean():.4f}')

# Plot posteriors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Density plot
axes[0].hist(control_samples, bins=50, alpha=0.6, label='Control', color=colors['primary'], density=True)
axes[0].hist(treatment_samples, bins=50, alpha=0.6, label='Treatment', color=colors['secondary'], density=True)
axes[0].axvline(control_samples.mean(), color=colors['primary'], linestyle='--', linewidth=2, label=f'Control Mean: {control_samples.mean():.4f}')
axes[0].axvline(treatment_samples.mean(), color=colors['secondary'], linestyle='--', linewidth=2, label=f'Treatment Mean: {treatment_samples.mean():.4f}')
axes[0].set_xlabel('Conversion Rate', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Posterior Distributions of Conversion Rates', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Difference distribution
diff_samples = treatment_samples - control_samples
axes[1].hist(diff_samples, bins=50, alpha=0.7, color=colors['success'])
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='No Difference')
axes[1].axvline(diff_samples.mean(), color=colors['danger'], linestyle='--', linewidth=2, label=f'Mean Diff: {diff_samples.mean():.4f}')
axes[1].set_xlabel('Treatment - Control', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title(f'Distribution of Difference (P(T>C) = {prob_treatment_better:.2%})', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_bayesian_posteriors.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPlot saved: nb03_bayesian_posteriors.png')

## Method 4: Bootstrap Resampling

### How It Works

Bootstrap is a **non-parametric method**: it does not assume any particular distribution. Instead, it asks "If we resample users from our observed data (with replacement), what range of treatment effects do we see?"

**The Approach:**
1. For each resample (e.g., 10,000 times):
   - Randomly draw users from the treatment group (with replacement) to the same size
   - Randomly draw users from the control group (with replacement) to the same size
   - Calculate conversion rate difference
2. Examine the distribution of these 10,000 differences
3. Take the 2.5th and 97.5th percentiles as the 95% CI
4. Count how many resamples had difference < 0 to estimate p-value

### Key Concepts

- **No distributional assumptions**: Works with any metric
- **Bootstrap CI**: Directly from the empirical distribution
- **Bootstrap p-value**: Proportion of resamples with effect in opposite direction

### Benefits
- Very flexible, works with any metric
- No distributional assumptions
- Empirically grounded (based on actual data)
- Easy to understand

### Limitations
- Requires larger sample sizes (need enough data to resample)
- Computationally intensive
- Bootstrap distribution depends on actual observed data

In [ ]:
# Bootstrap resampling
np.random.seed(42)

n_bootstrap = 10000
bootstrap_diffs = []

for i in range(n_bootstrap):
    # Resample each group
    control_boot = np.random.choice(control['converted'].values, size=len(control), replace=True)
    treatment_boot = np.random.choice(treatment['converted'].values, size=len(treatment), replace=True)
    
    # Calculate difference in conversion rates
    diff = treatment_boot.mean() - control_boot.mean()
    bootstrap_diffs.append(diff)

bootstrap_diffs = np.array(bootstrap_diffs)

# Bootstrap CI
bootstrap_ci_low = np.percentile(bootstrap_diffs, 2.5)
bootstrap_ci_high = np.percentile(bootstrap_diffs, 97.5)

print('=== BOOTSTRAP RESULTS ===')
print(f'Bootstrap resamples: {n_bootstrap}')
print(f'Mean difference: {bootstrap_diffs.mean():.4f}')
print(f'Std dev of differences: {bootstrap_diffs.std():.4f}')
print(f'95% CI: [{bootstrap_ci_low:.4f}, {bootstrap_ci_high:.4f}]')

# Bootstrap p-value: proportion of resamples where treatment <= control
bootstrap_pvalue = (bootstrap_diffs <= 0).mean()
print(f'\nBootstrap p-value (two-sided): {bootstrap_pvalue * 2:.6f}')
print(f'Result: {"SIGNIFICANT" if bootstrap_pvalue < 0.025 else "NOT SIGNIFICANT"}')

# Plot bootstrap distribution
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(bootstrap_diffs, bins=50, alpha=0.7, color=colors['primary'], edgecolor='black')
ax.axvline(0, color='red', linestyle='--', linewidth=2.5, label='No Effect')
ax.axvline(bootstrap_diffs.mean(), color=colors['secondary'], linestyle='--', linewidth=2.5, label=f'Mean: {bootstrap_diffs.mean():.4f}')
ax.axvline(bootstrap_ci_low, color=colors['success'], linestyle=':', linewidth=2, label=f'95% CI: [{bootstrap_ci_low:.4f}, {bootstrap_ci_high:.4f}]')
ax.axvline(bootstrap_ci_high, color=colors['success'], linestyle=':', linewidth=2)

ax.set_xlabel('Treatment - Control (Conversion Rate Difference)', fontsize=12)
ax.set_ylabel('Frequency (out of 10,000 resamples)', fontsize=12)
ax.set_title('Bootstrap Distribution of Treatment Effect', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_bootstrap_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPlot saved: nb03_bootstrap_distribution.png')

## Method 5: Sequential Testing (Optional Stopping)

### How It Works

Sequential testing lets you **peek at results during the experiment** and stop early if evidence is strong enough. Traditional methods assume you decide sample size upfront; sequential methods account for multiple looks.

**Conceptual Approach:**
1. As data comes in, recalculate p-value
2. Plot p-value over time (as you collect more users)
3. Stop if p-value crosses a threshold (e.g., 0.025 for 5% significance with corrected thresholds)
4. Controls "false positive" rate even with early peeks

### Key Concepts

- **Multiple Testing Problem**: Peeking multiple times inflates false positive rate
- **Spending Function**: Controls type-I error across all peeks
- **Early Stop**: Can stop and declare winner or loser before target N

### Benefits
- Can stop early and save costs
- More ethical (stop if treatment clearly better or worse)
- Accounts for peeking problem

### Limitations
- More complex to implement correctly
- Requires pre-defined stopping rule
- Loses some statistical power vs. fixed sample
- Easy to do wrong (peeking without adjustment inflates false positives!)

In [ ]:
# Sequential testing simulation
# Simulate monitoring as data accumulates

np.random.seed(42)

# Use proportions_ztest iteratively
cumulative_control_conversions = []
cumulative_control_n = []
cumulative_treatment_conversions = []
cumulative_treatment_n = []
cumulative_pvalues = []

# Simulate collecting data in batches
batch_size = 20
min_samples_per_group = 100  # Do not look until we have enough

for i in range(batch_size, len(control), batch_size):
    j = (i * len(treatment)) // len(control)  # Proportional to treatment
    
    cum_ctrl_conv = control['converted'].iloc[:i].sum()
    cum_ctrl_n = i
    cum_trt_conv = treatment['converted'].iloc[:j].sum()
    cum_trt_n = j
    
    if cum_ctrl_n >= min_samples_per_group and cum_trt_n >= min_samples_per_group:
        try:
            z, p = proportions_ztest(
                [cum_trt_conv, cum_ctrl_conv],
                [cum_trt_n, cum_ctrl_n],
                alternative='two-sided'
            )
            cumulative_pvalues.append(p)
            cumulative_control_n.append(cum_ctrl_n)
        except:
            pass

print('=== SEQUENTIAL TESTING SIMULATION ===')
print(f'Monitoring checkpoints: {len(cumulative_pvalues)}')
print(f'Sample size range: {cumulative_control_n[0] if cumulative_control_n else "N/A"} to {cumulative_control_n[-1] if cumulative_control_n else "N/A"} per group')
print(f'\nFirst 5 p-values:')
for i in range(min(5, len(cumulative_pvalues))):
    print(f'  n={cumulative_control_n[i]:4d}: p={cumulative_pvalues[i]:.4f}')
print(f'\nFinal p-value: {cumulative_pvalues[-1]:.6f}' if cumulative_pvalues else 'N/A')

# Plot cumulative p-values
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(cumulative_control_n, cumulative_pvalues, marker='o', linewidth=2.5, 
        markersize=6, color=colors['primary'], label='Observed p-value')
ax.axhline(0.05, color='red', linestyle='--', linewidth=2, label='alpha = 0.05 (traditional threshold)')
ax.axhline(0.025, color=colors['danger'], linestyle=':', linewidth=2, label='alpha = 0.025 (sequential corrected)')
ax.fill_between(cumulative_control_n, 0, 0.025, alpha=0.1, color='green', label='Strong evidence region')

ax.set_xlabel('Sample Size per Group', fontsize=12)
ax.set_ylabel('P-value', fontsize=12)
ax.set_title('Sequential Monitoring: Cumulative P-values Over Time', fontsize=13, fontweight='bold')
ax.set_ylim(-0.01, max(cumulative_pvalues) * 1.1)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_sequential_testing.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPlot saved: nb03_sequential_testing.png')

## Section 2: Revenue Analysis

Conversion rate is important, but revenue impact is what the business cares about. Revenue distributions are often **non-normal** (skewed, with outliers), so we use the **Mann-Whitney U test** (non-parametric alternative to t-test).

In [ ]:
# Revenue analysis
print('=== REVENUE STATISTICS ===')
print('\nControl group:')
print(f'  N: {len(control)}')
print(f'  Mean revenue: ${control["revenue"].mean():.2f}')
print(f'  Median revenue: ${control["revenue"].median():.2f}')
print(f'  Std dev: ${control["revenue"].std():.2f}')
print(f'  Min: ${control["revenue"].min():.2f}')
print(f'  Max: ${control["revenue"].max():.2f}')

print('\nTreatment group:')
print(f'  N: {len(treatment)}')
print(f'  Mean revenue: ${treatment["revenue"].mean():.2f}')
print(f'  Median revenue: ${treatment["revenue"].median():.2f}')
print(f'  Std dev: ${treatment["revenue"].std():.2f}')
print(f'  Min: ${treatment["revenue"].min():.2f}')
print(f'  Max: ${treatment["revenue"].max():.2f}')

print('\nDifference:')
print(f'  Mean lift: ${treatment["revenue"].mean() - control["revenue"].mean():.2f}')
print(f'  Lift %: {((treatment["revenue"].mean() / control["revenue"].mean()) - 1):.2%}')

# Mann-Whitney U test (non-parametric)
u_stat, p_value_mw = stats.mannwhitneyu(treatment['revenue'], control['revenue'], alternative='two-sided')

print(f'\n=== MANN-WHITNEY U TEST ===')
print(f'U-statistic: {u_stat:.2f}')
print(f'P-value: {p_value_mw:.6f}')
print(f'Result: {"SIGNIFICANT" if p_value_mw < 0.05 else "NOT SIGNIFICANT"}')
print(f'\nInterpretation: Revenue distribution is {"significantly different" if p_value_mw < 0.05 else "not significantly different"} between groups')

# Visualize revenue distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
data_to_plot = [control['revenue'], treatment['revenue']]
axes[0].boxplot(data_to_plot, labels=['Control', 'Treatment'], patch_artist=True,
                boxprops=dict(facecolor=colors['primary'], alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_ylabel('Revenue per User ($)', fontsize=12)
axes[0].set_title('Revenue Distribution: Boxplot', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Overlapping histograms
axes[1].hist(control['revenue'], bins=30, alpha=0.6, label=f'Control (mean=${control["revenue"].mean():.2f})', 
             color=colors['primary'], density=True)
axes[1].hist(treatment['revenue'], bins=30, alpha=0.6, label=f'Treatment (mean=${treatment["revenue"].mean():.2f})', 
             color=colors['secondary'], density=True)
axes[1].axvline(control['revenue'].mean(), color=colors['primary'], linestyle='--', linewidth=2)
axes[1].axvline(treatment['revenue'].mean(), color=colors['secondary'], linestyle='--', linewidth=2)
axes[1].set_xlabel('Revenue per User ($)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Revenue Distribution: Density', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_revenue_analysis.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPlot saved: nb03_revenue_analysis.png')

## Method Comparison: Summary Table

| **Method** | **Test Stat** | **Assumption** | **P-value** | **CI** | **When to Use** | **Strength** | **Limitation** |
|:---|:---|:---|:---:|:---:|:---|:---|:---|
| **Z-Test** | z-score | Normal (CLT) | Yes | Yes | Large samples, binary | Fast, intuitive, standard | Requires large N |
| **Chi-Square** | chi-square | Independence | Yes | No | Categorical data, 2x2 | No distribution assumption | Cannot compare magnitudes |
| **Bayesian** | Posterior | Prior + likelihood | No, credibility | Yes | Any metric, prior knowledge | Direct probability | Subjective prior |
| **Bootstrap** | Empirical | None | Yes | Yes | Any metric, robustness | No assumptions, flexible | Computationally heavy |
| **Sequential** | Z-test (repeated) | Normal + spending | Yes (corrected) | Yes | Monitoring, cost sensitive | Early stop, control FP | Complex setup, loses power |

### Quick Decision Guide

**Use Z-Test if:**
- You have large sample sizes
- Conversion rate is your metric
- You want speed and standard reporting

**Use Chi-Square if:**
- You have categorical data
- You want a distribution-free test

**Use Bayesian if:**
- You want to answer "probability treatment is better?"
- You have prior information to incorporate
- You are comfortable with Bayesian framework

**Use Bootstrap if:**
- You want robustness across any metric
- Your data might not be normally distributed
- You have sufficient sample size

**Use Sequential if:**
- You want to peek during the experiment
- Cost of running the experiment is high
- You need early stopping rules

## Key Findings & Recommendations

Based on the analysis above, here is what the data tells us:

## Guardrails & North Star: Where to Apply Them in A/B Testing

---

### North Star Connection

Every A/B test should be tied to a **North Star Metric** — otherwise you're optimizing in a vacuum. Before running any experiment, answer:

1. **What North Star does this experiment serve?** (e.g., "Purchase Conversion Rate")
2. **What is the hypothesis?** (e.g., "The new checkout flow will increase purchase rate by 3%")
3. **How does a win here move the North Star?** (e.g., "3% lift in conversion × 5,000 monthly visitors = ~150 additional purchases/month")

**Interview tip:** When someone presents you with an A/B test result, don't just look at the p-value. Ask: *"Does this move the North Star enough to matter?"* A statistically significant but tiny effect may not be worth the engineering cost to ship.

---

### Guardrail Metrics for Experiments

A/B tests can produce a "win" on the primary metric while causing hidden damage. **Guardrails catch these tradeoffs.**

| Primary Metric Being Tested | Guardrail to Monitor | Why |
|---|---|---|
| Conversion rate | Revenue per user | Higher conversion from discounts could lower average revenue |
| Pages viewed / engagement | Bounce rate, session errors | Engagement tricks (clickbait) can frustrate users |
| Time on site | Task completion rate | More time isn't always better — users might be confused |
| Feature adoption | Support tickets, error rates | A confusing new feature increases load on support |

**In healthcare (SmarterDx context):**
- Primary: AI suggestion acceptance rate
- Guardrails: False positive rate (AI shouldn't suggest incorrect diagnoses), physician time-on-task (shouldn't slow down workflows), compliance error rate, patient safety flags

---

### Automated Alerting Rules for Experiments

In practice, product teams set up **experiment guardrail monitors** that automatically flag or halt an experiment if:
- Any guardrail degrades by more than a predefined threshold (e.g., >2% drop in NPS)
- The guardrail metric crosses an absolute floor (e.g., false positive rate exceeds 5%)
- A safety metric shows a statistically significant negative change (even if the primary metric looks good)

**This is why sequential testing (Method 5) matters** — it lets you monitor continuously and stop early if guardrails are breached, rather than waiting until the experiment is "done."

---

### What Our Data Can and Can't Tell Us

With our A/B dataset, we can measure conversion rate and revenue — which covers the primary metric and one guardrail (revenue per user). We **cannot** measure satisfaction, error rates, or workflow disruption from this data, but in an interview you should always mention these as things you *would* monitor in a real experiment.

In [ ]:
# Summary statistics
control_conv = control['converted'].mean()
treatment_conv = treatment['converted'].mean()
lift = ((treatment_conv / control_conv) - 1) * 100

print('=== EXPERIMENT SUMMARY ===')
print(f'Sample size: {len(df):,} users ({len(control):,} control, {len(treatment):,} treatment)')
print(f'\nConversion Performance:')
print(f'  Control: {control_conv:.2%} ({control["converted"].sum():.0f} conversions)')
print(f'  Treatment: {treatment_conv:.2%} ({treatment["converted"].sum():.0f} conversions)')
print(f'  Lift: {lift:+.1f}%')

print(f'\nStatistical Significance:')
print(f'  Z-test p-value: {p_value_ztest:.6f} -> {"SIGNIFICANT" if p_value_ztest < 0.05 else "NOT SIGNIFICANT"}')
print(f'  Chi-square p-value: {p_value_chi2:.6f} -> {"SIGNIFICANT" if p_value_chi2 < 0.05 else "NOT SIGNIFICANT"}')
print(f'  Bootstrap p-value: {bootstrap_pvalue*2:.6f} -> {"SIGNIFICANT" if bootstrap_pvalue < 0.025 else "NOT SIGNIFICANT"}')
print(f'  Bayesian P(T>C): {prob_treatment_better:.2%}')

print(f'\nRevenue Impact:')
control_revenue = control['revenue'].mean()
treatment_revenue = treatment['revenue'].mean()
revenue_lift = ((treatment_revenue / control_revenue) - 1) * 100
print(f'  Control: ${control_revenue:.2f}/user')
print(f'  Treatment: ${treatment_revenue:.2f}/user')
print(f'  Lift: ${treatment_revenue - control_revenue:+.2f} ({revenue_lift:+.1f}%)')
print(f'  Mann-Whitney p-value: {p_value_mw:.6f} -> {"SIGNIFICANT" if p_value_mw < 0.05 else "NOT SIGNIFICANT"}')

print(f'\n=== RECOMMENDATION ===')
if p_value_ztest < 0.05 and treatment_conv > control_conv:
    print(f'LAUNCH TREATMENT: Significant positive effect on conversion.')
    print(f'  The treatment shows a {lift:.1f}% improvement in conversion rate.')
    if p_value_mw < 0.05:
        print(f'  BONUS: Revenue impact is also significant (+{revenue_lift:.1f}%).')
elif p_value_ztest < 0.05 and treatment_conv < control_conv:
    print(f'DO NOT LAUNCH: Significant NEGATIVE effect on conversion.')
elif p_value_ztest >= 0.05:
    print(f'INCONCLUSIVE: No statistically significant difference detected.')
    print(f'  Observed lift: {lift:+.1f}%, but likely due to random variation.')
    print(f'  Either increase sample size or accept the risk of launching no-effect change.')

## Conclusion

This notebook demonstrated **5 different statistical methods** for analyzing A/B tests:

1. **Frequentist Z-Test**: The industry standard for binary metrics. Fast, intuitive, requires large samples.
2. **Chi-Square Test**: Distribution-free test of independence. Good for categorical data.
3. **Bayesian A/B Testing**: Directly answers "is treatment better?" with probability. Requires prior specification.
4. **Bootstrap**: Flexible, non-parametric, works with any metric. Computationally intensive.
5. **Sequential Testing**: Enables peeking and early stopping while controlling false positives.

### Why Multiple Methods?

- Different methods have different **assumptions and strengths**
- Results should be **consistent** across methods (if not, investigate)
- Some methods answer **different questions** (p-value vs. probability treatment is better)
- Understanding tradeoffs makes you a better analyst

### Next Steps

- In production, pick one primary method and stick with it
- Use multiple methods for **validation** and robustness checks
- Always check **practical significance**: is the lift meaningful for business?
- Consider **cost of error**: false positive vs. false negative
- Account for **multiple comparisons** if testing multiple metrics